# Финальное решение: модель отклика на кредитный оффер


Структура решения:

1. загрузка данных и базовая проверка;
2. построение признаков для заявки и оффера;
3. обучение нескольких моделей градиентного бустинга на фиксированных фолдах;
4. ранговое усреднение предсказаний;
5. формирование финального submission.

Во всех признаках с использованием таргета применяется out-of-fold схема, чтобы не переносить целевую переменную из validation-фолдов в обучение.


In [ ]:
from pathlib import Path

DATA_DIR    = Path(".")
TRAIN_PATH  = DATA_DIR / "train_apps.csv"
TEST_PATH   = DATA_DIR / "test_apps.csv"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"
OUT_DIR     = DATA_DIR

ID, TARGET, DATE = "front_id", "target_value", "decision_day"
SEED = 42

FAST_MODE = False


RUN_CORE_STACK     = True    # базовый стек
RUN_ENHANCED_STACK = True    # расширенный стек
RUN_LGB_GBDT = True
RUN_LGB_DART = True          # фиксированное число деревьев
RUN_CAT      = True
RUN_XGB      = True


ADD_GLOBAL_RANKS       = True
ADD_STABLE_BINS        = True
ADD_MISSING_SIGNATURE  = True
ADD_WITHIN_REQUEST     = True
ADD_EXTENDED_TE        = True
ADD_PAST_HISTORY_TE    = True
ADD_EXTRA_INTERACTIONS = True


N_FOLDS = 5
if FAST_MODE:
    SEEDS_LGB = [42]
    SEEDS_XGB = [42]
    SEEDS_CAT = [42]
    N_ESTIMATORS = 2500
    DART_ESTIMATORS = 900
    EARLY_STOP = 120
else:
    SEEDS_LGB = [42, 2025, 777]
    SEEDS_XGB = [42, 2025]
    SEEDS_CAT = [42, 2025]
    N_ESTIMATORS = 6500
    DART_ESTIMATORS = 1800
    EARLY_STOP = 300

LR = 0.02
print("Config loaded. FAST_MODE =", FAST_MODE)

In [ ]:
import os, gc, json, math, random, warnings, hashlib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

random.seed(SEED)
np.random.seed(SEED)

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression


def _try(name):
    try:
        return __import__(name), True
    except Exception as e:
        print(f"[warn] {name} недоступен: {e}")
        return None, False

lgb, HAS_LGB = _try("lightgbm")
xgb, HAS_XGB = _try("xgboost")
try:
    from catboost import CatBoostClassifier, Pool
    HAS_CAT = True
except Exception as e:
    print(f"[warn] catboost недоступен: {e}")
    HAS_CAT = False

RUN_LGB_GBDT = RUN_LGB_GBDT and HAS_LGB
RUN_LGB_DART = RUN_LGB_DART and HAS_LGB
RUN_XGB = RUN_XGB and HAS_XGB
RUN_CAT = RUN_CAT and HAS_CAT

print(f"LightGBM={HAS_LGB} | CatBoost={HAS_CAT} | XGBoost={HAS_XGB}")
assert RUN_LGB_GBDT or RUN_CAT or RUN_XGB, "Не найдена ни одна библиотека бустинга."

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
sample    = pd.read_csv(SAMPLE_PATH)


def parse_dt(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_datetime("2024-01-01") + pd.to_timedelta(s.astype(float), unit="D")
    return pd.to_datetime(s, errors="coerce")

train_raw["_dt"] = parse_dt(train_raw[DATE])
test_raw["_dt"]  = parse_dt(test_raw[DATE])

print("train :", train_raw.shape)
print("test  :", test_raw.shape)
print("sample:", sample.shape)
print("target mean:", round(float(train_raw[TARGET].mean()), 6))
print("train dates:", train_raw["_dt"].min(), "→", train_raw["_dt"].max())
print("test  dates:", test_raw["_dt"].min(),  "→", test_raw["_dt"].max())

sample_ids = set(sample[ID])
known_sample_mask = train_raw[ID].isin(sample_ids).values
n_known_sample = int(known_sample_mask.sum())
print(f"\nsample ids in test : {int(sample[ID].isin(test_raw[ID]).sum())}")
print(f"sample ids in train: {n_known_sample}")
if n_known_sample:
    kd = train_raw.loc[known_sample_mask, "_dt"]
    print(f"known-sample train dates: {kd.min().date()} → {kd.max().date()}")

y = train_raw[TARGET].astype(int).values


##  Диагнстика данных

В этом разделе проверяется динамика целевой переменной по времени и выделяется дополнительный контрольный срез. Он используется только для диагностики качества блендов и не участвует в обучении как таргетная информация для теста.


In [ ]:
month_tbl = (train_raw.assign(_month=train_raw["_dt"].dt.to_period("M").astype(str))
             .groupby("_month")[TARGET]
             .agg(n="size", target_rate="mean"))
display(month_tbl)

if n_known_sample >= 1000 and len(np.unique(y[known_sample_mask])) == 2:
    ANCHOR_MASK = known_sample_mask.copy()
    ANCHOR_NAME = "sample_train_anchor"
else:
    # fallback: самый свежий хвост train
    cutoff = train_raw["_dt"].quantile(0.88)
    ANCHOR_MASK = (train_raw["_dt"] >= cutoff).values
    ANCHOR_NAME = f"date_tail_from_{cutoff.date()}"

RECENT_MASK = (train_raw["_dt"] >= pd.Timestamp("2025-03-01")).values
print(f"ANCHOR = {ANCHOR_NAME}: rows={ANCHOR_MASK.sum()}, target_rate={y[ANCHOR_MASK].mean():.5f}")
print(f"RECENT 2025-03+: rows={RECENT_MASK.sum()}, target_rate={y[RECENT_MASK].mean():.5f}")

In [ ]:
def quick_adversarial_auc(cols, tag):
    if not HAS_LGB or len(cols) == 0:
        return np.nan, None
    X = pd.concat([train_raw[cols], test_raw[cols]], ignore_index=True)
    yy = np.r_[np.zeros(len(train_raw)), np.ones(len(test_raw))]
    idx = np.arange(len(X))
    rng = np.random.default_rng(SEED)
    rng.shuffle(idx)
    tr_idx, va_idx = idx[:len(idx)//2], idx[len(idx)//2:]
    m = lgb.LGBMClassifier(
        n_estimators=350, learning_rate=0.05, num_leaves=31,
        min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1, verbosity=-1,
    )
    m.fit(X.iloc[tr_idx], yy[tr_idx], eval_set=[(X.iloc[va_idx], yy[va_idx])], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])
    p = m.predict_proba(X.iloc[va_idx])[:, 1]
    auc = roc_auc_score(yy[va_idx], p)
    imp = pd.DataFrame({"feature": cols, "importance": m.feature_importances_}).sort_values("importance", ascending=False)
    print(f"adversarial AUC [{tag}] = {auc:.5f}")
    display(imp.head(12))
    return auc, imp

raw_num_cols = [c for c in train_raw.columns
                if c not in {ID, TARGET, DATE, "_dt"} and pd.api.types.is_numeric_dtype(train_raw[c])]
macro_like = {"cb_rate", "offered_rate"}
client_num_cols = [c for c in raw_num_cols if c not in macro_like]

adv_all_auc, adv_all_imp = quick_adversarial_auc(raw_num_cols, "all numeric")
adv_client_auc, adv_client_imp = quick_adversarial_auc(client_num_cols, "client numeric without cb/offered")

## Feature engineering

Используются два набора признаков:

- `core`: базовые признаки заявки, лимитов, ставки, активности и пропусков;
- `enhanced`: дополнительные сегменты, сглаженные target/history encodings, глобальные ранги и признаки контекста оффера.

Календарные признаки считаются только как свойства даты. Нормализации внутри месяцев/недель по объединению `train+test` не используются.


In [ ]:
DROP = {ID, TARGET, DATE, "_dt"}
ALL_COLS = [c for c in train_raw.columns if c not in DROP]
CAT_COLS = [c for c in ["db_group_last", "fl_adminarea"] if c in train_raw.columns]
NUM_COLS = [c for c in ALL_COLS if c not in CAT_COLS and pd.api.types.is_numeric_dtype(train_raw[c])]
print("CAT_COLS:", CAT_COLS)
print("NUM_COLS:", len(NUM_COLS))


def safe_div(a, b):
    if isinstance(b, pd.Series):
        b = b.replace(0, np.nan)
    elif b == 0:
        b = np.nan
    return a / b


def reduce_mem(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype("float32")
        elif pd.api.types.is_integer_dtype(df[c]) and c not in {ID, TARGET}:
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


def add_basic_features(df):
    """Базовые признаки: дата, экономика оффера, лимиты, динамика активности, логи и пропуски."""
    df = df.copy()
    d = df["_dt"]

   
    df["dd_month"]     = d.dt.month.astype("float32")
    df["dd_day"]       = d.dt.day.astype("float32")
    df["dd_dow"]       = d.dt.dayofweek.astype("float32")
    df["dd_quarter"]   = d.dt.quarter.astype("float32")
    df["dd_week"]      = d.dt.isocalendar().week.astype("float32")
    df["dd_is_mend"]   = d.dt.is_month_end.astype("int8")
    df["dd_is_mstart"] = d.dt.is_month_start.astype("int8")
    df["dd_month_sin"] = np.sin(2*np.pi*df["dd_month"]/12).astype("float32")
    df["dd_month_cos"] = np.cos(2*np.pi*df["dd_month"]/12).astype("float32")
    df["dd_dow_sin"]   = np.sin(2*np.pi*df["dd_dow"]/7).astype("float32")
    df["dd_dow_cos"]   = np.cos(2*np.pi*df["dd_dow"]/7).astype("float32")
    df["dd_days_since"] = (d - pd.Timestamp("2024-01-01")).dt.days.astype("float32")

    
    df["n_missing"] = df[ALL_COLS].isna().sum(axis=1).astype("float32")
    for c in ["fl_hdb_bki_total_active_products", "days_from_authperson_registration",
              "balance_rur_amt_30_min", "p75_time_spent_minutes",
              "loan_rev_max_start_non_fin", "cnt_cred_loan_90", "sum_deb_ul_90"]:
        if c in df.columns:
            df[f"{c}_isna"] = df[c].isna().astype("int8")

    # экономика оффера относительно ключевой ставки
    if {"offered_rate", "cb_rate"}.issubset(df.columns):
        df["rate_spread"]      = df["offered_rate"] - df["cb_rate"]
        df["rate_spread_abs"]  = df["rate_spread"].abs()
        df["rate_ratio"]       = safe_div(df["offered_rate"], df["cb_rate"])
        df["rate_log_spread"]  = np.log1p(df["offered_rate"].clip(lower=0)) - np.log1p(df["cb_rate"].clip(lower=0))

    # лимиты и позиция запрошенной суммы
    if {"overdraft_limit_min", "overdraft_limit_max"}.issubset(df.columns):
        df["limit_mid"]   = (df["overdraft_limit_min"] + df["overdraft_limit_max"]) / 2
        df["limit_width"] = df["overdraft_limit_max"] - df["overdraft_limit_min"]
        df["limit_width_rel"] = safe_div(df["limit_width"], df["limit_mid"].abs() + 1)
        df["max_to_min"]  = safe_div(df["overdraft_limit_max"], df["overdraft_limit_min"].abs() + 1)
        if "loan_amount_last" in df.columns:
            df["loan_to_min"]  = safe_div(df["loan_amount_last"], df["overdraft_limit_min"].abs() + 1)
            df["loan_to_max"]  = safe_div(df["loan_amount_last"], df["overdraft_limit_max"].abs() + 1)
            df["loan_to_mid"]  = safe_div(df["loan_amount_last"], df["limit_mid"].abs() + 1)
            df["loan_minus_max"] = df["loan_amount_last"] - df["overdraft_limit_max"]
            df["loan_minus_mid"] = df["loan_amount_last"] - df["limit_mid"]
            df["loan_pos_in_limit"] = safe_div(df["loan_amount_last"] - df["overdraft_limit_min"],
                                                df["limit_width"].abs() + 1)
            df["req_above_max"] = (df["loan_amount_last"] > df["overdraft_limit_max"]).astype("int8")
            df["req_below_min"] = (df["loan_amount_last"] < df["overdraft_limit_min"]).astype("int8")

    
    for s30, s90 in [("sum_deb_ul_30", "sum_deb_ul_90"),
                     ("cnt_deb_ul_ip_30", "cnt_deb_ul_ip_90")]:
        if {s30, s90}.issubset(df.columns):
            df[f"{s30}_share"]   = safe_div(df[s30], df[s90].abs() + 1)
            df[f"{s30}_60extra"] = df[s90] - df[s30]
            df[f"{s30}_accel"]   = safe_div(df[s30] * 3, df[s90].abs() + 1)

   
    money_like = ["loan_amount_last", "overdraft_limit_min", "overdraft_limit_max",
                  "sum_deb_ul_30", "sum_deb_ul_90", "sum_deb_investment_90",
                  "balance_rur_amt_30_min", "limit_mid", "limit_width"]
    for c in money_like:
        if c in df.columns:
            df[f"{c}_log"] = np.sign(df[c]) * np.log1p(df[c].abs())

   
    if {"count_all_corp_dashboard_events", "p75_time_spent_minutes"}.issubset(df.columns):
        df["events_per_min"] = safe_div(df["count_all_corp_dashboard_events"],
                                        df["p75_time_spent_minutes"].abs() + 1)
    if {"fl_hdb_bki_total_active_products", "corp_credit_products"}.issubset(df.columns):
        df["bki_vs_corp_products"] = df["fl_hdb_bki_total_active_products"] - df["corp_credit_products"]
    return df

In [ ]:
def add_freq_encoding(tr, te, cols):
    tr, te = tr.copy(), te.copy()
    cols = [c for c in cols if c in tr.columns and c in te.columns]
    if not cols:
        return tr, te
    both = pd.concat([tr[cols], te[cols]], ignore_index=True)
    n = len(both)
    for c in cols:
        key_all = both[c].astype("object").fillna("__NA__").astype(str)
        vc = key_all.value_counts(dropna=False)
        freq = (vc / n).to_dict(); cnt = vc.to_dict()
        for part in (tr, te):
            k = part[c].astype("object").fillna("__NA__").astype(str)
            part[f"{c}_freq"]  = k.map(freq).fillna(0).astype("float32")
            part[f"{c}_count"] = k.map(cnt).fillna(0).astype("float32")
    return tr, te


def add_group_stats(tr, te, group_cols, value_cols):
    tr, te = tr.copy(), te.copy()
    group_cols = [g for g in group_cols if g in tr.columns and g in te.columns]
    value_cols = [v for v in value_cols if v in tr.columns and v in te.columns]
    both = pd.concat([tr, te], ignore_index=True)
    for g in group_cols:
        gk = both[g].astype("object").fillna("__NA__").astype(str)
        for v in value_cols:
            grp = both.assign(_k=gk).groupby("_k")[v]
            mean_map = grp.mean().to_dict()
            std_map  = grp.std().to_dict()
            med_map  = grp.median().to_dict()
            for part in (tr, te):
                k = part[g].astype("object").fillna("__NA__").astype(str)
                mean_s = k.map(mean_map).astype("float32")
                std_s  = k.map(std_map).replace(0, np.nan).astype("float32")
                med_s  = k.map(med_map).astype("float32")
                part[f"{v}_mean_by_{g}"] = mean_s
                part[f"{v}_std_by_{g}"]  = std_s
                part[f"{v}_diff_mean_{g}"] = (part[v] - mean_s).astype("float32")
                part[f"{v}_diff_median_{g}"] = (part[v] - med_s).astype("float32")
                part[f"{v}_z_{g}"] = ((part[v] - mean_s) / (std_s + 1e-6)).astype("float32")
    return tr, te


def add_global_ranks(tr, te, cols):
    tr, te = tr.copy(), te.copy()
    cols = [c for c in cols if c in tr.columns and c in te.columns]
    for c in cols:
        pooled = pd.concat([tr[c], te[c]], ignore_index=True)
        r = pooled.rank(pct=True, na_option="keep")
        tr[f"{c}_grank"] = r.iloc[:len(tr)].values.astype("float32")
        te[f"{c}_grank"] = r.iloc[len(tr):].values.astype("float32")
    return tr, te


def add_pooled_bins(tr, te, cols, n_bins=10):
    tr, te = tr.copy(), te.copy()
    made = []
    for c in cols:
        if c not in tr.columns or c not in te.columns:
            continue
        pooled = pd.concat([tr[c], te[c]], ignore_index=True)
        non_na = pooled.dropna()
        if non_na.nunique() < 3:
            continue
        qs = np.unique(np.nanquantile(non_na, np.linspace(0, 1, n_bins + 1)))
        if len(qs) < 3:
            continue
        qs[0] = -np.inf; qs[-1] = np.inf
        b = pd.cut(pooled, bins=qs, labels=False, include_lowest=True)
        b = b.astype("float").fillna(-1).astype(int).astype(str)
        name = f"{c}_bin"
        tr[name] = b.iloc[:len(tr)].values
        te[name] = b.iloc[len(tr):].values
        made.append(name)
    return tr, te, made


def make_key(df, cols, name):
    use = [c for c in cols if c in df.columns]
    if not use:
        df[name] = "__NA__"
    else:
        df[name] = df[use].astype("object").fillna("__NA__").astype(str).agg("|".join, axis=1)
    return df


def add_missing_signature(tr, te):
    tr, te = tr.copy(), te.copy()
    miss_cols = [c for c in ["loan_rev_max_start_non_fin", "fl_hdb_bki_total_active_products",
                             "app_term_mean_360", "balance_rur_amt_30_min", "p75_time_spent_minutes",
                             "sum_deb_ul_90", "cnt_cred_loan_90", "days_from_authperson_registration"]
                 if c in tr.columns]
    if not miss_cols:
        return tr, te, []
    for part in (tr, te):
        part["missing_sig"] = part[miss_cols].isna().astype(int).astype(str).agg("".join, axis=1)
    return tr, te, ["missing_sig"]


def add_within_request_features(tr, te):
    
    tr, te = tr.copy(), te.copy()
    both = pd.concat([tr.assign(_src=0), te.assign(_src=1)], ignore_index=True)
    offer_cols = [c for c in ["offered_rate", "overdraft_limit_min", "overdraft_limit_max", "loan_amount_last"] if c in both.columns]
    client_cols = [c for c in NUM_COLS if c not in offer_cols and c not in {"cb_rate"} and c in both.columns]
    key_cols = client_cols + [c for c in CAT_COLS if c in both.columns]
    if not key_cols or not offer_cols:
        return tr, te, 0.0
    # округление снижает шум от float, но не создает календарных bucket-нормализаций
    kdf = both[key_cols].copy()
    for c in kdf.columns:
        if pd.api.types.is_numeric_dtype(kdf[c]):
            kdf[c] = kdf[c].round(4)
    req_key = kdf.astype("object").fillna("∅").astype(str).agg("|".join, axis=1) + "@" + both["_dt"].dt.strftime("%Y-%m-%d")
    both["_req"] = pd.factorize(req_key)[0]
    sizes = both.groupby("_req")["_req"].transform("size")
    both["req_size"] = sizes.astype("float32")
    both["req_is_multi"] = (sizes > 1).astype("int8")
    for c in offer_cols:
        g = both.groupby("_req")[c]
        both[f"{c}_rank_in_req"] = g.rank(pct=True).astype("float32")
        both[f"{c}_minus_reqmin"] = (both[c] - g.transform("min")).astype("float32")
        both[f"{c}_minus_reqmax"] = (both[c] - g.transform("max")).astype("float32")
    if "offered_rate" in offer_cols:
        g = both.groupby("_req")["offered_rate"]
        both["is_cheapest_in_req"] = (both["offered_rate"] <= g.transform("min") + 1e-9).astype("int8")
        both["rate_spread_in_req"] = (g.transform("max") - g.transform("min")).astype("float32")
    new_cols = [c for c in both.columns if c.endswith("_in_req") or c.endswith("_minus_reqmin") or c.endswith("_minus_reqmax") or c in ["req_size", "req_is_multi", "is_cheapest_in_req", "rate_spread_in_req"]]
    tr_add = both.loc[both["_src"] == 0, new_cols].reset_index(drop=True)
    te_add = both.loc[both["_src"] == 1, new_cols].reset_index(drop=True)
    for c in new_cols:
        tr[c] = tr_add[c].values
        te[c] = te_add[c].values
    return tr, te, float(both["req_is_multi"].mean())

In [ ]:
# Фолды фиксируются один раз: и для OOF target encoding, и для моделей.
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_id = np.full(len(train_raw), -1, dtype=int)
for f, (_, va) in enumerate(skf.split(train_raw, y)):
    fold_id[va] = f
FOLDS = [(np.where(fold_id != f)[0], np.where(fold_id == f)[0]) for f in range(N_FOLDS)]
print("fold sizes:", [len(va) for _, va in FOLDS])


def oof_target_encode(tr, te, cols, smoothing=80.0, noise=0.0, seed=SEED):
    tr, te = tr.copy(), te.copy()
    cols = [c for c in cols if c in tr.columns and c in te.columns]
    if not cols:
        return tr, te
    rng = np.random.default_rng(seed)
    gmean = float(y.mean())
    for c in cols:
        key = tr[c].astype("object").fillna("__NA__").astype(str).values
        oof = np.empty(len(tr), dtype="float64")
        for f in range(N_FOLDS):
            tr_mask = fold_id != f
            va_mask = fold_id == f
            df_in = pd.DataFrame({"k": key[tr_mask], "y": y[tr_mask]})
            agg = df_in.groupby("k")["y"].agg(["sum", "count"])
            enc = (agg["sum"] + gmean * smoothing) / (agg["count"] + smoothing)
            oof[va_mask] = pd.Series(key[va_mask]).map(enc).fillna(gmean).values
        if noise > 0:
            oof = oof * (1 + rng.normal(0, noise, len(oof)))
        tr[f"{c}_te"] = oof.astype("float32")

        df_all = pd.DataFrame({"k": key, "y": y})
        agg = df_all.groupby("k")["y"].agg(["sum", "count"])
        enc = (agg["sum"] + gmean * smoothing) / (agg["count"] + smoothing)
        te[f"{c}_te"] = te[c].astype("object").fillna("__NA__").astype(str).map(enc).fillna(gmean).astype("float32")
    return tr, te


def add_past_history_te(tr, te, cols, smoothing=120.0, min_count_for_raw=2):
    """Временной historical TE: для train каждая строка видит только прошлые строки, для test — весь train.
    Это имитирует доступную на момент теста историю и часто сильнее обычного OOF TE при повторяющихся сегментах.
    """
    tr, te = tr.copy(), te.copy()
    cols = [c for c in cols if c in tr.columns and c in te.columns]
    if not cols:
        return tr, te
    gmean = float(y.mean())
    order = np.argsort(tr["_dt"].values.astype("datetime64[ns]"))
    for c in cols:
        keys = tr[c].astype("object").fillna("__NA__").astype(str).values
        sums, cnts = {}, {}
        enc = np.empty(len(tr), dtype="float32")
        cnt_past = np.zeros(len(tr), dtype="float32")
        for i in order:
            k = keys[i]
            s = sums.get(k, 0.0)
            n = cnts.get(k, 0)
            enc[i] = (s + gmean * smoothing) / (n + smoothing)
            cnt_past[i] = n
            sums[k] = s + y[i]
            cnts[k] = n + 1
        tr[f"{c}_past_te"] = enc
        tr[f"{c}_past_count_log"] = np.log1p(cnt_past).astype("float32")

        df_all = pd.DataFrame({"k": keys, "y": y}).groupby("k")["y"].agg(["sum", "count"])
        enc_all = (df_all["sum"] + gmean * smoothing) / (df_all["count"] + smoothing)
        cnt_all = df_all["count"]
        kk = te[c].astype("object").fillna("__NA__").astype(str)
        te[f"{c}_past_te"] = kk.map(enc_all).fillna(gmean).astype("float32")
        te[f"{c}_past_count_log"] = np.log1p(kk.map(cnt_all).fillna(0)).astype("float32")
    return tr, te

In [ ]:
def build_feature_tables(mode="core"):
    assert mode in {"core", "enhanced"}
    extra = (mode == "enhanced")
    tr = add_basic_features(train_raw)
    te = add_basic_features(test_raw)

   
    for c in CAT_COLS:
        tr[c] = tr[c].astype("object").fillna("__NA__").astype(str)
        te[c] = te[c].astype("object").fillna("__NA__").astype(str)

    
    tr, te = add_freq_encoding(tr, te, CAT_COLS)
    group_values = [c for c in ["offered_rate", "rate_spread", "loan_amount_last",
                                "overdraft_limit_max", "loan_to_max", "sum_deb_ul_90",
                                "balance_rur_amt_30_min", "cb_rate"] if c in tr.columns]
    tr, te = add_group_stats(tr, te, CAT_COLS, group_values)

    te_cols = list(CAT_COLS)
    hist_cols = []
    key_cols_for_freq = []

    if extra:
        if ADD_GLOBAL_RANKS:
            rank_src = [c for c in ["offered_rate", "rate_spread", "rate_ratio", "loan_to_max",
                                    "loan_pos_in_limit", "limit_width", "overdraft_limit_max",
                                    "loan_amount_last", "n_missing"] if c in tr.columns]
            tr, te = add_global_ranks(tr, te, rank_src)

        if ADD_STABLE_BINS:
            bin_src = [c for c in ["days_from_authperson_registration", "fl_hdb_bki_total_active_products",
                                   "count_all_corp_dashboard_events", "p75_time_spent_minutes",
                                   "loan_amount_last", "overdraft_limit_max", "rate_spread", "loan_to_max",
                                   "limit_width", "cnt_cred_loan_90", "sum_deb_ul_90",
                                   "balance_rur_amt_30_min", "n_missing"] if c in tr.columns]
            tr, te, made_bins = add_pooled_bins(tr, te, bin_src, n_bins=10)
        else:
            made_bins = []

        if ADD_MISSING_SIGNATURE:
            tr, te, miss_keys = add_missing_signature(tr, te)
        else:
            miss_keys = []

        # interaction categorical keys for TE/count only
        for part in (tr, te):
            if set(CAT_COLS).issubset(part.columns) and len(CAT_COLS) >= 2:
                make_key(part, CAT_COLS, "key_area_group")
            if "db_group_last" in part.columns and "rate_spread_bin" in part.columns:
                make_key(part, ["db_group_last", "rate_spread_bin"], "key_group_ratebin")
            if "fl_adminarea" in part.columns and "loan_to_max_bin" in part.columns:
                make_key(part, ["fl_adminarea", "loan_to_max_bin"], "key_area_loanbin")
            if "db_group_last" in part.columns and "days_from_authperson_registration_bin" in part.columns:
                make_key(part, ["db_group_last", "days_from_authperson_registration_bin"], "key_group_authbin")
            if "fl_adminarea" in part.columns and "fl_hdb_bki_total_active_products_bin" in part.columns:
                make_key(part, ["fl_adminarea", "fl_hdb_bki_total_active_products_bin"], "key_area_bkibin")
            if "missing_sig" in part.columns and "db_group_last" in part.columns:
                make_key(part, ["db_group_last", "missing_sig"], "key_group_misssig")

        constructed_keys = [c for c in ["key_area_group", "key_group_ratebin", "key_area_loanbin",
                                        "key_group_authbin", "key_area_bkibin", "key_group_misssig"]
                            if c in tr.columns]
        key_cols_for_freq = made_bins + miss_keys + constructed_keys
        tr, te = add_freq_encoding(tr, te, key_cols_for_freq)

        if ADD_EXTENDED_TE:
            te_cols = te_cols + key_cols_for_freq + constructed_keys
        
            te_cols = list(dict.fromkeys(te_cols))

        if ADD_PAST_HISTORY_TE:
            hist_cols = [c for c in ["key_area_group", "key_group_authbin", "key_area_bkibin", "missing_sig"]
                         if c in tr.columns]

        if ADD_WITHIN_REQUEST:
            tr, te, multi_share = add_within_request_features(tr, te)
            print(f"[{mode}] pseudo-request multi share = {multi_share:.4f}")

        if ADD_EXTRA_INTERACTIONS:
            for part in (tr, te):
                if {"rate_spread", "loan_to_max"}.issubset(part.columns):
                    part["rate_x_loan_to_max"] = (part["rate_spread"] * part["loan_to_max"]).astype("float32")
                if {"rate_spread", "limit_width_rel"}.issubset(part.columns):
                    part["rate_x_limit_width_rel"] = (part["rate_spread"] * part["limit_width_rel"]).astype("float32")
                if {"sum_deb_ul_30_share", "rate_spread"}.issubset(part.columns):
                    part["activity_share_x_rate"] = (part["sum_deb_ul_30_share"] * part["rate_spread"]).astype("float32")
                if {"events_per_min", "loan_to_max"}.issubset(part.columns):
                    part["events_per_min_x_loan_to_max"] = (part["events_per_min"] * part["loan_to_max"]).astype("float32")

    
    tr, te = oof_target_encode(tr, te, te_cols, smoothing=90.0 if extra else 80.0, noise=0.002 if extra else 0.0)
    if extra and hist_cols:
        tr, te = add_past_history_te(tr, te, hist_cols, smoothing=140.0)

   
    for c in CAT_COLS:
        if c in tr.columns:
            tr[c] = tr[c].astype("object").fillna("__NA__").astype(str)
            te[c] = te[c].astype("object").fillna("__NA__").astype(str)

    for df in (tr, te):
        df.replace([np.inf, -np.inf], np.nan, inplace=True)

    raw_cat_set = set(CAT_COLS)
    features = []
    for c in tr.columns:
        if c in DROP or c not in te.columns:
            continue
        if c in raw_cat_set:
            features.append(c)
        elif pd.api.types.is_numeric_dtype(tr[c]) and tr[c].notna().any() and tr[c].nunique(dropna=True) > 1:
            features.append(c)
    # deduplicate
    features = list(dict.fromkeys(features))
    tr = reduce_mem(tr)
    te = reduce_mem(te)
    print(f"[{mode}] train_fe={tr.shape}, test_fe={te.shape}, features={len(features)}, cat={CAT_COLS}")
    return tr, te, features, list(CAT_COLS)

train_core, test_core, FEATURES_CORE, CAT_CORE = build_feature_tables("core")
if RUN_ENHANCED_STACK:
    train_enh, test_enh, FEATURES_ENH, CAT_ENH = build_feature_tables("enhanced")
else:
    train_enh, test_enh, FEATURES_ENH, CAT_ENH = None, None, [], []

##  Обучение моделей

Финальный ансамбль строится через несколько моделей и несколько seed'ов. Для устойчивости используется fold-bagging, усреднение по seed'ам и ранговое усреднение предсказаний разных моделей.


In [ ]:
def rank01(x):
    return pd.Series(x).rank(pct=True).values.astype("float32")


def prep_lgb(train_fe, test_fe, features, cat_cols):
    X = train_fe[features].copy(); Xt = test_fe[features].copy()
    cats = [c for c in cat_cols if c in features]
    for c in cats:
        X[c] = X[c].astype("category")
        Xt[c] = Xt[c].astype("category")
    return X, Xt, cats


def prep_num(train_fe, test_fe, features, cat_cols):
    feats = [c for c in features if c not in set(cat_cols)]
    return train_fe[feats].copy(), test_fe[feats].copy(), feats


def prep_cat(train_fe, test_fe, features, cat_cols):
    X = train_fe[features].copy(); Xt = test_fe[features].copy()
    cats = [c for c in cat_cols if c in features]
    for c in cats:
        X[c] = X[c].astype(str)
        Xt[c] = Xt[c].astype(str)
    cat_idx = [features.index(c) for c in cats]
    return X, Xt, cats, cat_idx


def model_slice_metrics(pred):
    out = {"auc_all": roc_auc_score(y, pred)}
    for nm, mask in [(ANCHOR_NAME, ANCHOR_MASK), ("recent_2025_03p", RECENT_MASK)]:
        if mask.sum() >= 500 and len(np.unique(y[mask])) == 2:
            out[f"auc_{nm}"] = roc_auc_score(y[mask], pred[mask])
        else:
            out[f"auc_{nm}"] = np.nan
    return out

In [ ]:
def run_lgb_gbdt(train_fe, test_fe, features, cat_cols, seeds, name):
    X, Xt, cats = prep_lgb(train_fe, test_fe, features, cat_cols)
    oof = np.zeros(len(X), dtype="float32")
    test = np.zeros(len(Xt), dtype="float32")
    seed_scores, best_iters = [], []
    for sd in seeds:
        oof_s = np.zeros(len(X), dtype="float32")
        test_s = np.zeros(len(Xt), dtype="float32")
        params = dict(objective="binary", metric="auc", boosting_type="gbdt",
                      learning_rate=LR, n_estimators=N_ESTIMATORS, num_leaves=64,
                      min_child_samples=70, subsample=0.82, subsample_freq=1,
                      colsample_bytree=0.78, reg_lambda=9.0, reg_alpha=0.3,
                      random_state=sd, n_jobs=-1, verbosity=-1)
        for tr_idx, va_idx in FOLDS:
            m = lgb.LGBMClassifier(**params)
            m.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[va_idx], y[va_idx])], eval_metric="auc",
                  categorical_feature=cats,
                  callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
            oof_s[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
            test_s += m.predict_proba(Xt)[:, 1].astype("float32") / len(FOLDS)
            best_iters.append(m.best_iteration_ or N_ESTIMATORS)
        oof += rank01(oof_s) / len(seeds)
        test += rank01(test_s) / len(seeds)
        seed_scores.append(roc_auc_score(y, oof_s))
    auc = roc_auc_score(y, oof)
    print(f"[{name}] seed AUCs={np.round(seed_scores, 6).tolist()} | rank AUC={auc:.6f} | mean_iter={int(np.mean(best_iters))}")
    return {"name": name, "oof": oof, "test": test, "auc": auc, "family": "lgb_gbdt"}


def run_lgb_dart(train_fe, test_fe, features, cat_cols, seeds, name):
    X, Xt, cats = prep_lgb(train_fe, test_fe, features, cat_cols)
    oof = np.zeros(len(X), dtype="float32")
    test = np.zeros(len(Xt), dtype="float32")
    seed_scores = []
    for sd in seeds:
        oof_s = np.zeros(len(X), dtype="float32")
        test_s = np.zeros(len(Xt), dtype="float32")
        params = dict(objective="binary", metric="auc", boosting_type="dart",
                      learning_rate=0.025, n_estimators=DART_ESTIMATORS, num_leaves=48,
                      min_child_samples=90, subsample=0.80, subsample_freq=1,
                      colsample_bytree=0.75, reg_lambda=12.0, reg_alpha=0.7,
                      random_state=sd, n_jobs=-1, verbosity=-1)
        for tr_idx, va_idx in FOLDS:
            m = lgb.LGBMClassifier(**params)
            # DART + early stopping обычно шумит, поэтому фиксированное число деревьев.
            m.fit(X.iloc[tr_idx], y[tr_idx], categorical_feature=cats)
            oof_s[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
            test_s += m.predict_proba(Xt)[:, 1].astype("float32") / len(FOLDS)
        oof += rank01(oof_s) / len(seeds)
        test += rank01(test_s) / len(seeds)
        seed_scores.append(roc_auc_score(y, oof_s))
    auc = roc_auc_score(y, oof)
    print(f"[{name}] seed AUCs={np.round(seed_scores, 6).tolist()} | rank AUC={auc:.6f}")
    return {"name": name, "oof": oof, "test": test, "auc": auc, "family": "lgb_dart"}


def run_xgb_model(train_fe, test_fe, features, cat_cols, seeds, name):
    X, Xt, feats = prep_num(train_fe, test_fe, features, cat_cols)
    oof = np.zeros(len(X), dtype="float32")
    test = np.zeros(len(Xt), dtype="float32")
    seed_scores = []
    for sd in seeds:
        oof_s = np.zeros(len(X), dtype="float32")
        test_s = np.zeros(len(Xt), dtype="float32")
        for tr_idx, va_idx in FOLDS:
            m = xgb.XGBClassifier(
                n_estimators=N_ESTIMATORS, learning_rate=LR, max_depth=6,
                min_child_weight=5, subsample=0.82, colsample_bytree=0.78,
                reg_lambda=8.0, reg_alpha=0.4, gamma=0.0,
                tree_method="hist", eval_metric="auc", early_stopping_rounds=EARLY_STOP,
                random_state=sd, n_jobs=-1,
            )
            m.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[va_idx], y[va_idx])], verbose=False)
            oof_s[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
            test_s += m.predict_proba(Xt)[:, 1].astype("float32") / len(FOLDS)
        oof += rank01(oof_s) / len(seeds)
        test += rank01(test_s) / len(seeds)
        seed_scores.append(roc_auc_score(y, oof_s))
    auc = roc_auc_score(y, oof)
    print(f"[{name}] seed AUCs={np.round(seed_scores, 6).tolist()} | rank AUC={auc:.6f}")
    return {"name": name, "oof": oof, "test": test, "auc": auc, "family": "xgb"}


def run_cat_model(train_fe, test_fe, features, cat_cols, seeds, name):
    X, Xt, cats, cat_idx = prep_cat(train_fe, test_fe, features, cat_cols)
    test_pool = Pool(Xt, cat_features=cat_idx)
    oof = np.zeros(len(X), dtype="float32")
    test = np.zeros(len(Xt), dtype="float32")
    seed_scores = []
    for sd in seeds:
        oof_s = np.zeros(len(X), dtype="float32")
        test_s = np.zeros(len(Xt), dtype="float32")
        for tr_idx, va_idx in FOLDS:
            m = CatBoostClassifier(
                iterations=N_ESTIMATORS, learning_rate=LR, depth=6,
                l2_leaf_reg=8.0, random_strength=0.8, bagging_temperature=0.5,
                loss_function="Logloss", eval_metric="AUC",
                od_type="Iter", od_wait=EARLY_STOP, random_seed=sd,
                verbose=False, allow_writing_files=False,
            )
            m.fit(Pool(X.iloc[tr_idx], y[tr_idx], cat_features=cat_idx),
                  eval_set=Pool(X.iloc[va_idx], y[va_idx], cat_features=cat_idx), use_best_model=True)
            oof_s[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
            test_s += m.predict_proba(test_pool)[:, 1].astype("float32") / len(FOLDS)
        oof += rank01(oof_s) / len(seeds)
        test += rank01(test_s) / len(seeds)
        seed_scores.append(roc_auc_score(y, oof_s))
    auc = roc_auc_score(y, oof)
    print(f"[{name}] seed AUCs={np.round(seed_scores, 6).tolist()} | rank AUC={auc:.6f}")
    return {"name": name, "oof": oof, "test": test, "auc": auc, "family": "cat"}

In [ ]:
models = []

def add_stack(prefix, train_fe, test_fe, features, cat_cols):
    before = len(models)
    print("\n" + "="*100)
    print(f"RUN STACK: {prefix} | n_features={len(features)}")
    if RUN_LGB_GBDT:
        models.append(run_lgb_gbdt(train_fe, test_fe, features, cat_cols, SEEDS_LGB, name=f"{prefix}_lgb_gbdt"))
    if RUN_LGB_DART:
        models.append(run_lgb_dart(train_fe, test_fe, features, cat_cols, SEEDS_LGB[:max(1, min(2, len(SEEDS_LGB)))], name=f"{prefix}_lgb_dart"))
    if RUN_CAT:
        models.append(run_cat_model(train_fe, test_fe, features, cat_cols, SEEDS_CAT, name=f"{prefix}_cat"))
    if RUN_XGB:
        models.append(run_xgb_model(train_fe, test_fe, features, cat_cols, SEEDS_XGB, name=f"{prefix}_xgb"))
    print(f"added {len(models)-before} models")
    gc.collect()

if RUN_CORE_STACK:
    add_stack("core", train_core, test_core, FEATURES_CORE, CAT_CORE)
if RUN_ENHANCED_STACK:
    add_stack("enh", train_enh, test_enh, FEATURES_ENH, CAT_ENH)

summary = []
for m in models:
    row = {"model": m["name"], "family": m["family"], **model_slice_metrics(m["oof"])}
    summary.append(row)
summary = pd.DataFrame(summary).sort_values("auc_all", ascending=False)
display(summary)


## Делаем submission

Для финального решения используется небольшое число интерпретируемых блендов. Основной вариант — равномерное ранговое усреднение всех обученных моделей.


In [ ]:
assert models, "Нет обученных моделей."

def mean_rank(models_subset):
    o = np.column_stack([m["oof"] for m in models_subset]).mean(axis=1)
    t = np.column_stack([m["test"] for m in models_subset]).mean(axis=1)
    return rank01(o), rank01(t)

candidates = {}
# single модели
for m in models:
    candidates[f"single_{m['name']}"] = (rank01(m["oof"]), rank01(m["test"]))

core_models = [m for m in models if m["name"].startswith("core_")]
enh_models  = [m for m in models if m["name"].startswith("enh_")]
if core_models:
    candidates["core_equal"] = mean_rank(core_models)
if enh_models:
    candidates["enh_equal"] = mean_rank(enh_models)
candidates["all_equal"] = mean_rank(models)

if core_models and enh_models:
    core_o, core_t = candidates["core_equal"]
    enh_o, enh_t = candidates["enh_equal"]
    for w_core in [0.80, 0.70, 0.60, 0.50]:
        name = f"guard_core{int(w_core*100)}_enh{int((1-w_core)*100)}"
        candidates[name] = (rank01(w_core*core_o + (1-w_core)*enh_o),
                            rank01(w_core*core_t + (1-w_core)*enh_t))

if len(models) >= 2:
    oof_mat = np.column_stack([m["oof"] for m in models])
    test_mat = np.column_stack([m["test"] for m in models])
    meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    meta = LogisticRegression(C=0.4, max_iter=3000)
    try:
        stack_oof = cross_val_predict(meta, oof_mat, y, cv=meta_cv, method="predict_proba")[:, 1]
        meta.fit(oof_mat, y)
        stack_test = meta.predict_proba(test_mat)[:, 1]
        candidates["meta_logreg_oof"] = (rank01(stack_oof), rank01(stack_test))
        print("meta weights:", dict(zip([m["name"] for m in models], np.round(meta.coef_[0], 4))))
    except Exception as e:
        print("[warn] meta stack failed:", e)

rows = []
for name, (oof_pred, test_pred) in candidates.items():
    row = {"candidate": name, **model_slice_metrics(oof_pred)}
    rows.append(row)
blend_report = pd.DataFrame(rows)

metric_cols = [c for c in blend_report.columns if c.startswith("auc_")]
for c in metric_cols:
    blend_report[f"rank_{c}"] = blend_report[c].rank(pct=True, na_option="bottom")

anchor_col = f"auc_{ANCHOR_NAME}"
rank_anchor = f"rank_{anchor_col}"
rank_recent = "rank_auc_recent_2025_03p"
rank_all = "rank_auc_all"
weights = []
if rank_anchor in blend_report.columns and not blend_report[anchor_col].isna().all():
    weights.append((rank_anchor, 0.50))
if rank_recent in blend_report.columns and not blend_report["auc_recent_2025_03p"].isna().all():
    weights.append((rank_recent, 0.25))
weights.append((rank_all, 0.25))

blend_report["selection_score"] = 0.0
for c, w in weights:
    blend_report["selection_score"] += w * blend_report[c]
blend_report = blend_report.sort_values(["selection_score", "auc_all"], ascending=False)
display(blend_report[["candidate", "auc_all", anchor_col, "auc_recent_2025_03p", "selection_score"]].head(25))

BEST_CANDIDATE = blend_report.iloc[0]["candidate"]
print("BEST_CANDIDATE =", BEST_CANDIDATE)

In [ ]:
def make_submission(test_pred, filename):
    pred_map = dict(zip(test_raw[ID].values, np.asarray(test_pred).astype(float)))
    known_map = dict(zip(train_raw[ID].values, train_raw[TARGET].values))

    sub = sample[[ID]].copy()
    sub[TARGET] = sub[ID].map(pred_map)

  
    mask_known = sub[ID].isin(known_map.keys())
    sub.loc[mask_known, TARGET] = sub.loc[mask_known, ID].map(known_map)

    fallback = float(np.nanmean(test_pred))
    if not np.isfinite(fallback):
        fallback = float(train_raw[TARGET].mean())

    sub[TARGET] = (
        sub[TARGET]
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(fallback)
        .clip(0, 1)
    )

    out_path = OUT_DIR / filename
    sub.to_csv(out_path, index=False)

    print(
        f"{filename}: rows={len(sub)}, "
        f"known_overwritten={int(mask_known.sum())}, "
        f"range=[{sub[TARGET].min():.4f}, {sub[TARGET].max():.4f}], "
        f"mean={sub[TARGET].mean():.6f}"
    )
    return sub


# Финальный выбранный бленд.
FINAL_CANDIDATE = "all_equal"

if FINAL_CANDIDATE not in candidates:
    raise KeyError(
        f"Не найден candidate={FINAL_CANDIDATE}. "
        f"Доступные candidates: {list(candidates.keys())}"
    )

subs = {}


subs["submission_final.csv"] = make_submission(
    candidates[FINAL_CANDIDATE][1],
    "submission_final.csv",
)


optional_candidates = {
    "submission_final_proxy_selected.csv": BEST_CANDIDATE,
    "submission_guard_core60_enh40.csv": "guard_core60_enh40",
    "submission_guard_core70_enh30.csv": "guard_core70_enh30",
    "submission_core_equal.csv": "core_equal",
    "submission_enhanced_equal.csv": "enh_equal",
    "submission_best_single_proxy.csv": (
        blend_report[blend_report["candidate"].str.startswith("single_")]
        .sort_values(["selection_score", "auc_all"], ascending=False)
        .iloc[0]["candidate"]
    ),
}

for filename, cand in optional_candidates.items():
    if cand in candidates:
        subs[filename] = make_submission(candidates[cand][1], filename)


blend_report.to_csv(OUT_DIR / "blend_report.csv", index=False)
summary.to_csv(OUT_DIR / "model_report.csv", index=False)

print("\nFINAL_CANDIDATE =", FINAL_CANDIDATE)
print("Главный файл для отправки организаторам: submission_final.csv")
display(subs["submission_final.csv"].head())


In [ ]:
if HAS_LGB:
    try:
        use_enh = RUN_ENHANCED_STACK and len(FEATURES_ENH) > 0
        tr_imp = train_enh if use_enh else train_core
        te_imp = test_enh if use_enh else test_core
        feats_imp = FEATURES_ENH if use_enh else FEATURES_CORE
        cats_imp = CAT_ENH if use_enh else CAT_CORE
        Ximp, Xtimp, cats_lgb = prep_lgb(tr_imp, te_imp, feats_imp, cats_imp)
        tr_idx, va_idx = FOLDS[0]
        m = lgb.LGBMClassifier(objective="binary", metric="auc", boosting_type="gbdt",
                               learning_rate=0.03, n_estimators=1500, num_leaves=64,
                               min_child_samples=70, subsample=0.82, colsample_bytree=0.78,
                               reg_lambda=9.0, reg_alpha=0.3, random_state=SEED,
                               n_jobs=-1, verbosity=-1)
        m.fit(Ximp.iloc[tr_idx], y[tr_idx], eval_set=[(Ximp.iloc[va_idx], y[va_idx])], eval_metric="auc",
              categorical_feature=cats_lgb, callbacks=[lgb.early_stopping(100, verbose=False)])
        imp = pd.DataFrame({"feature": feats_imp, "gain": m.booster_.feature_importance(importance_type="gain"),
                            "split": m.booster_.feature_importance(importance_type="split")}).sort_values("gain", ascending=False)
        display(imp.head(40))
        imp.to_csv(OUT_DIR / "feature_importance_lgb.csv", index=False)
    except Exception as e:
        print("feature importance skipped:", e)

## Итоговое использование

Основной файл для отправки:

`submission_final.csv`

Ключевые детали решения:

- основа — ансамбль LightGBM, CatBoost и XGBoost;
- используются два набора признаков: базовый и расширенный;
- target encoding построен через out-of-fold схему;
- групповые статистики и frequency-признаки считаются без использования `target_value`;
- если `sample_submission.csv` содержит `front_id`, уже присутствующий в train, для него подставляется известный `target_value`.
